
Code to convert MATLAB processing into Python

Created for dlx56_mPFC_1p_SohalLab repo
based off code in ruleshifting-inscopix private repo

#### Import packages


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations
import os
import sys
from pathlib import Path
from typing import List, Any, Dict
import numpy as np
import pandas as pd

import shutil
import time
from datetime import datetime

#concurrent/parallel processing
from joblib import Parallel, delayed  # from functools import partial #for specific type of parallel deployment

#load preprocess specific functions
import notebook_setup
info = notebook_setup.setup()
from run_manifest import create_run_manifest, update_run_database
from config_preprocessing import load_config
from analysis_config_loader import load_analysis_config  # new import, personal config 

preprocess_func_folder = Path(info["repo_root"]) / "preprocess_functions"
sys.path.append(str(preprocess_func_folder)) #add preprocess_functions folder to path to load modules
from trial_detection import return_trial_num_at_frame, build_phase_masks, get_trial_stage_map, get_inRS_isError
from shuffle_methods import random_roll_columns_numba, random_roll_columns_fast, create_shuffled_df_optimized, random_roll_columns, get_mean_postoutcome_stage_activity
from detect_cell_enrichment import get_cell_stage_enrichment, get_cell_ensemble_info_per_subject
from matlab_obj_to_python import load_matlab_object, check_corrupted_files, truncate_post_outcome_to_15s, label_frame_sections_df, add_task_stage_to_raster_df

# set up plotting code
import matplotlib as matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Apply mplstyle via absolute path from setup info
style_path = Path(info["function_py_storage"]) / "paper_plot.mplstyle"
print(f"Loading style guide at {style_path}")
if style_path.is_file():
    plt.style.use(str(style_path))
else:
    print(f"[warn] Style not found at: {style_path}")

#### Define file config variables and analysis parameters ####

In [ ]:
# set up locations for input/output 
root_dir = Path(r'c:\\Users\\13car\\Dropbox\\local_github_repos_personal\\dlx56_mPFC_1p_SohalLab\\code')
os.chdir(root_dir)

#import local yaml for env preset variables
config_path = Path(r"analysis_config.yaml")  # adjust as needed
analysis_config = load_analysis_config(config_path)
stage_names = analysis_config.task_phase_names
print(analysis_config)

# Load general file configuration
config = load_config('config.yaml')
data_type = config['data']['data_type_used']
preprocessing = config['preprocessing']
shuffle_cfg = config['shuffles']
print(config)

# Create unique run ID
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
START_TIME = time.time()
print(f"Run ID: {RUN_ID}, Started: {datetime.now()}")

# Create output directory with run ID
results_dir = config['data']['results_dir']
run_output_dir = results_dir / 'shuffles' / f"shuffle_run_{RUN_ID}"
run_output_dir.mkdir(parents=True, exist_ok=True)

#set data location
data_dir = config['data']['data_dir']
source_dataset_location= config['data']['source_dataset_location']

#hardcoded/old links for checking
hardcode_results_dir = Path(r"C:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results")
hardcode_data_dir = Path(r'c:\\Users\\13car\\Dropbox\\local_github_repos_personal\\dlx56_mPFC_1p_SohalLab\\data')
hardcode_source_dataset_location = Path(r"C:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\data\dataset_objects_24-Nov-2024_hour_19") #source dataset location has files created 10-26-2024

#print config paths for verification
print(f"\nFolders used in run ID: {RUN_ID}")
print(f"Results dir: {results_dir}")
print(f"Data dir: {data_dir}")
print(f"Source dataset location: {source_dataset_location}")

# Copy config to output directory for reference
shutil.copy('config.yaml', run_output_dir / 'config_used.yaml')

In [ ]:
#set parameters
data_types = ['raster', 'dff']
data_type_used = data_types[0] #OR, 'dff'
run_activity_enrichment = True
#normalization
normalize = config['preprocessing']['normalization']
if data_type_used== 'raster':
    normalize = 'none'
print(f"Applying normalization: {normalize} on {data_type_used} data")


In [ ]:
#get all dataset names
content_names = [f for f in source_dataset_location.glob('**/*') if "object" in f.name] #verify you are using the correct number of dataset 
print(f"Found {len(content_names)} files in {source_dataset_location}")
assert len(content_names) == 35, "Number of datasets does not match expected"
# Load all files with error handling
loaded_objects = []
failed_files = []

for num, file in enumerate(content_names):
    obj = load_matlab_object(file)
    if obj is None:
        failed_files.append(file.name)
    else:
        loaded_objects.append(obj)
        print(f"Loading file {num}/{len(content_names)}: {obj['name']}: {obj['raster'].shape[0]} cells × {obj['raster'].shape[1]} frames. {obj['dff'].shape} temporal weights ")
print(f"\n SUMMARY: Successfully loaded {len(loaded_objects)}/{len(content_names)} files")
if failed_files: check_corrupted_files(loaded_objects, failed_files)

#### Extract and Transform Matlab Objects into stage-labeled dataframes

In [ ]:
from preprocessing_utils import dataset_obj_to_df, transform_all_datasets, get_intercol_corrs, read_shuffle_parquet_from_folder, pivot_trial_windows, extract_trial_windows, create_subject_trial_tseries_df
### testing
test_df = dataset_obj_to_df(loaded_objects[0],config, analysis_config,  datatype = 'raster', normalize = 'none') #test 1 dataset
print(test_df.info())
print(test_df.sample())

In [ ]:
loaded_objects[0].keys()

In [ ]:
# Loop over imported subject rasters and create pandas DataFrames
raster_dataframes= transform_all_datasets(loaded_objects,config, analysis_config, datatype = data_type_used, normalize = normalize)

#### Circular shuffle of frame labels 

In [ ]:
# Option 1: Global seed sequence across all subjects
run_shuffles = False  #set to True to run shuffles
n_shuf_per_subject = config['shuffles']['n_shuffles_per_subject'] #default = 5000
## Configuration #200 minutes for 5000 shuffles/subject on 14 cores
n_jobs=  config['shuffles']['n_cores'] -2 #base =  psutil.cpu_count(logical=False)-2  #leave 2 cores free
base_seed = config['shuffles']['base_seed'] #default: 42
n_subjects = len(raster_dataframes)

print(f"Total subjects: {n_subjects}, Shuffles per subject: {n_shuf_per_subject}")
print(f"Total shuffles to create: {n_subjects * n_shuf_per_subject}")

In [ ]:
#set shuffle storage data folder
run_output_folder = results_dir / Path(f"{data_type_used}_{normalize}_norm_ensemble_detection_{datetime.now().strftime('%d-%b-%Y')}_{n_shuf_per_subject} shuffles")
print(f" saving to {run_output_folder}")
run_output_folder.mkdir(parents=True, exist_ok=True)
shuffle_storage_folder = run_output_folder/ Path(f"shuffled_dfs_{n_shuf_per_subject}_per_subject")
print(f"storing shuffles in {shuffle_storage_folder}") 
shuffle_storage_folder.mkdir(parents=True, exist_ok=True)

#set RNG info 
rng_master = np.random.default_rng(base_seed) # Generate random seeds once at the start
seed_min, seed_max = 0, 2**31 # Generate N random seeds (each will be different)

## main shuffle body
all_subject_shuffles = {} # Process each subject with their allocated seed range
if run_shuffles:
    for subject_idx, (subject_name, subject_raster_df) in enumerate(raster_dataframes.items()):
        print(f"\nProcessing subject {subject_idx+1}/{n_subjects}: {subject_name}")
        cell_col = [c for c in subject_raster_df.columns if 'cell' in c]
        subject_seeds = rng_master.integers(0,seed_max, size=n_shuf_per_subject) # Allocate unique seed range for this subject
        # Run parallel shuffles for this subject- oputputs lists
        output_shuff = Parallel(n_jobs=n_jobs, backend="loky", verbose=1)(delayed(create_shuffled_df_optimized)((subject_raster_df, cell_col, int(seed))) for seed in subject_seeds)
        #concat and save files 
        all_shuffle_means = pd.concat(output_shuff).drop(0).reset_index('trial_section', drop = True)
        all_shuffle_means = all_shuffle_means.assign(**{'date_saved': datetime.now().strftime("%Y-%m-%d %H:%M"),
                                                        'n_shuffles': n_shuf_per_subject, 'subject_name': subject_name}) 
        all_subject_shuffles[subject_name] = all_shuffle_means
        output_shuff.to_parquet(shuffle_storage_folder / Path(f"{subject_name}_shuffled_mean_activity.parquet"))   #save file
        print(f" Completed + saved {len(output_shuff)} shuffles")
    print(f"\nAll {n_subjects} subjects processed. Saving shuffled mean activity DFs to {shuffle_storage_folder}")        

#### Extract parquet files and save run record

In [ ]:
def log_shuffle_run(results_dir: Path, shuffle_storage_folder: Path, 
                    n_shuf_per_subject: int, n_subjects: int, base_seed: int) -> None:
    """Log a completed shuffle run to the run log CSV.
    Parameters:
    - results_dir: Directory containing the run log
    - shuffle_storage_folder: Folder where shuffle parquet files were saved
    - n_shuf_per_subject: Number of shuffles per subject
    - n_subjects: Number of subjects processed
    - base_seed: Random seed used
    """
    shuffle_run_log_path = results_dir / "shuffle_run_log.csv"
    run_info = {'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),'shuffle_folder': str(shuffle_storage_folder),'n_shuffles_per_subject': n_shuf_per_subject,'n_subjects': n_subjects,'base_seed': base_seed}
    # Load existing log or create new one
    if shuffle_run_log_path.exists():
        run_log_df = pd.read_csv(shuffle_run_log_path)
        run_log_df = pd.concat([run_log_df, pd.DataFrame([run_info])], ignore_index=True)
    else:
        run_log_df = pd.DataFrame([run_info])
    # Save updated log
    run_log_df.to_csv(shuffle_run_log_path, index=False)
    print(f"Saved latest shuffle run to {shuffle_run_log_path}")

def load_latest_shuffle_run(results_dir: Path, shuffle_storage_folder= None):
    """    Load shuffles from the most recent run in the log.
    Parameters: results_dir: Directory containing the run log
    (Optional) shuffle_storage_folder=
    
    Returns:
    - shuffle_storage_folder: Path to the shuffle folder
    - all_subject_shuffles: Loaded shuffle data
    - latest_run: Series with run metadata (timestamp, n_shuffles, etc.)
    
    Raises: FileNotFoundError: If no shuffle run log exists    """

    shuffle_run_log_path = results_dir / "shuffle_run_log.csv"
    if not shuffle_run_log_path.exists():
        raise FileNotFoundError(f"No shuffle run log found at {shuffle_run_log_path}")    
    # Load log and get most recent run
    run_log_df = pd.read_csv(shuffle_run_log_path)
    latest_run = run_log_df.iloc[-1]
    if shuffle_storage_folder == None:
        shuffle_storage_folder = Path(latest_run['shuffle_folder'])

    print(f"Loading shuffles from Timestamp: {latest_run['timestamp']} | {latest_run['n_shuffles_per_subject']} Shuffles")
    print(f"  Folder: {shuffle_storage_folder}")
    # Load all parquet files from that folder
    all_subject_shuffles = read_shuffle_parquet_from_folder(shuffle_storage_folder)
    return shuffle_storage_folder, all_subject_shuffles, latest_run

if run_shuffles: # After generating shuffles...
    log_shuffle_run(results_dir, shuffle_storage_folder, n_shuf_per_subject, len(raster_dataframes), base_seed)
else:
    shuffle_storage_folder, all_subject_shuffles, latest_run = load_latest_shuffle_run(results_dir)

#### Given shuffle array, run enrichment detection

In [ ]:
#get + save all ensembles
all_ensembles = get_cell_ensemble_info_per_subject(raster_dataframes, all_subject_shuffles, n_shuf_per_subject)
ens_matrix = all_ensembles.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = 'enriched', fill_value = False).astype(float)
#optional- save ensemble info as parquet
if 'data_type' not in all_ensembles.columns:# if data_type col isn't present in the ensemble matrix, add it
    all_ensembles['data_type'] = data_type_used
new_python_ens_file = run_output_folder / Path(f"{data_type_used}_postoutcome_python_ensembles_{n_shuf_per_subject}_shuff_{datetime.now().strftime('%d-%b-%Y_')}.parquet")
all_ensembles.to_parquet(new_python_ens_file)

#### Within python latest shuffle run, compare within chunk stability


In [ ]:
def chunk_shuffles_by_subject(all_subject_shuffles: Dict[str, pd.DataFrame], raster_dataframes: Dict[str, pd.DataFrame], chunk_size: int = 1000) -> pd.DataFrame:
    """ Chunk shuffles by subject into smaller sets for enrichment analysis.
    Inputs:
    all_subject_shuffles : Dict of subject_name to shuffled mean activity DataFrame
    raster_dataframes : Dict of subject_name to raster DataFrame
    chunk_size : number of unique seeds per chunk
    Returns: pd.DataFrame of combined enrichment results across all subject-chunks"""
    # Separate shuffles into chunks of 1000 unique seeds PER SUBJECT
    print(f"Chunking shuffles by subject (chunk_size={chunk_size})")
    chunked_ensemble_results = []
    # Loop over each subject
    for subject_name, shuffle_df in all_subject_shuffles.items():
        # Get unique seeds for THIS subject
        subject_unique_seeds = sorted(shuffle_df['shuffle_seed'].unique())
        n_seeds = len(subject_unique_seeds)
        n_chunks = n_seeds // chunk_size
        print(f"\n=== Processing subject: {subject_name}. total seeds: {n_seeds}, {n_chunks} chunks ===")
        
        # Get corresponding raster data
        subject_raster_df = raster_dataframes[subject_name]
        cell_col = [c for c in subject_raster_df.columns if c.startswith('cell_')]
        
        # Loop over chunks for this subject
        for chunk_idx in range(n_chunks):
            # Get specific seeds for this chunk
            start_idx = chunk_idx * chunk_size
            end_idx = (chunk_idx + 1) * chunk_size
            chunk_seeds = subject_unique_seeds[start_idx:end_idx]
            chunk_shuffle_df = shuffle_df[shuffle_df['shuffle_seed'].isin(chunk_seeds)] # Filter shuffles for this chunk
            # Run enrichment analysis for this subject-chunk
            chunk_ensembles = get_cell_stage_enrichment(all_shuffle_means=chunk_shuffle_df, raster_df=subject_raster_df, cell_col=cell_col,n_shuf_per_subject=len(chunk_seeds))
            # Add metadata
            chunked_ensemble_results.append(chunk_ensembles.assign(**{'subject_name': subject_name,'chunk_index': chunk_idx, 'chunk_seed_min': chunk_seeds[0],'chunk_seed_max': chunk_seeds[-1],'n_seeds_in_chunk': len(chunk_seeds)}))

    # Combine all subject-chunk results
    all_chunked_ensembles = pd.concat(chunked_ensemble_results, ignore_index=True)
    print(f"Processed {all_chunked_ensembles['subject_name'].nunique()} Subjects into {all_chunked_ensembles.shape} shaped DF")
    print(f"Chunks per subject: {all_chunked_ensembles.groupby('subject_name')['chunk_index'].max() + 1}")
    return all_chunked_ensembles

## pivot chunked ensembles activation and plot heatmap
def get_pivoted_enrichment_heatmap(all_chunked_ensembles: pd.DataFrame, pivot_values = 'enriched') -> pd.DataFrame:
    """ Pivot chunked ensemble enrichment results and plot heatmap of correlations.
    Inputs: all_chunked_ensembles : DataFrame with columns ['subject_name', 'task_stage', 'cell', 'chunk_index', 'enriched']
    Returns: Pivoted DataFrame of enrichment by chunk"""

    enriched_by_chunk = all_chunked_ensembles.pivot_table(index=['subject_name', 'task_stage', 'cell'], columns='chunk_index',values=pivot_values,aggfunc='first')
    enriched_by_chunk.columns = [f'chunk_{i}' for i in enriched_by_chunk.columns]
    enriched_by_chunk=enriched_by_chunk.reset_index()
    return enriched_by_chunk

In [ ]:
all_chunked_ensembles = chunk_shuffles_by_subject(all_subject_shuffles, raster_dataframes, chunk_size=1000)

#### import raw shuffle data for DFF and spikes, to compare shuffle chunk similarity

In [ ]:
spikes_shuffle_path = results_dir / Path(f"spikes_ensemble_detection_20-Nov-2025_5000 shuffles/shuffled_dfs_5000_per_subject")
print(f"Loading shuffles from {spikes_shuffle_path}")
all_shuffles_spikes =  read_shuffle_parquet_from_folder(spikes_shuffle_path) # Load all parquet files from that folder
#if you are importing something that's NOT the current data type used, make sure to re-create raster dataframes for that datatype
# Loop over imported objects by subject and create pandas DataFrames, containing RASTERS
raster_dataframes_spikes = transform_all_datasets(loaded_objects, config, analysis_config, datatype = 'raster', normalize = 'none')
all_chunk_ens_spikes = chunk_shuffles_by_subject(all_shuffles_spikes, raster_dataframes_spikes, chunk_size=1000)
enriched_by_chunk_spikes = get_pivoted_enrichment_heatmap(all_chunk_ens_spikes)
enriched_by_chunk_spikes.head()


In [ ]:
raster_dataframes_spikes

In [ ]:
def plot_stage_correlation_heatmaps(enrich_compared_df: pd.DataFrame, 
                                   stage_names: list, 
                                   merge_chunk_col: list,
                                   value_compared: str,
                                   axes: np.ndarray,
                                   grid_shape: tuple = (2, 3),
                                   vmin: float = 0.5,
                                   cmap: str = 'viridis') -> None:
    """Plot correlation heatmaps for each task stage on a subplot grid."""
    nrows, ncols = grid_shape
    
    for s, stage in enumerate(stage_names):
        ax = axes.flatten()[s]
        row, col = s // ncols, s % ncols
        
        # Compute and plot correlation for this stage
        stage_vals = enrich_compared_df[enrich_compared_df['task_stage'] == stage][merge_chunk_col].corr()
        sns.heatmap(stage_vals, vmin=vmin, cmap=cmap, ax=ax)
        
        # Remove ticks if not on edges
        if col != 0: ax.set_yticks([])
        if row != nrows - 1: ax.set_xticks([])
        
        ax.set_title(f"{stage.replace('_', ' ')}: {value_compared} correlations")

In [ ]:
# #merge enrich by chunk spikes and enriched by chunk dff
enriched_by_chunk_dff = get_pivoted_enrichment_heatmap(all_chunked_ensembles)
merged_enriched_chunks = pd.merge(enriched_by_chunk_dff, enriched_by_chunk_spikes, on=['subject_name', 'task_stage', 'cell'], suffixes=('_dff', '_spikes'))
# Plot correlation heatmaps
value_compared = "membership"
fig, axes = plt.subplots(2, 3, figsize=(6,3.5))
plot_stage_correlation_heatmaps(enrich_compared_df=merged_enriched_chunks, stage_names=stage_names, merge_chunk_col=[c for c in merged_enriched_chunks.columns if 'chunk_' in c], value_compared=value_compared, axes = axes)
fig.suptitle(f"Python| Correlation of ensemble {value_compared} vectors of zscored DFF & Spike rasters. (Chunk = 1000 shuffles each)")
# Save figure
fig_filename = f"dff_vs_spikes_ensemble_{value_compared}_vector_chunkcorrs.jpg"
fig.savefig(results_dir / fig_filename, dpi=300, bbox_inches='tight')
print(f"Figure saved: {fig_filename} | In: {results_dir}")

In [ ]:
#use pvalue for corr 
pval_by_chunk_spikes = get_pivoted_enrichment_heatmap(all_chunk_ens_spikes, pivot_values = 'percent_shuffle >= real')
pval_by_chunk_dff = get_pivoted_enrichment_heatmap(all_chunked_ensembles, pivot_values = 'percent_shuffle >= real')
merge_pval_chunks = pd.merge(pval_by_chunk_dff, pval_by_chunk_spikes, on=['subject_name', 'task_stage', 'cell'], suffixes=('_dff', '_spikes'))

# Plot correlation heatmaps
value_compared = "p-value"
fig, axes = plt.subplots(2, 3, figsize=(6,3.5))
plot_stage_correlation_heatmaps(enrich_compared_df=merge_pval_chunks, stage_names=stage_names, merge_chunk_col=[c for c in merge_pval_chunks.columns if 'chunk_' in c], value_compared=value_compared, vmin = 0.65, axes = axes)
fig.suptitle(f"Python| Correlation of {value_compared} vs shuffle for activity of zscored DFF & Spike rasters. (Chunk = 1000 shuffles each)")
# Save figure
fig_filename = f"dff_vs_spikes_ensemble_{value_compared}_vector_chunkcorrs.jpg"
fig.savefig(results_dir / fig_filename, dpi=300, bbox_inches='tight')
print(f"Figure saved: {fig_filename} | In: {results_dir}")

#### import both DFF & spike ensemble parquets


In [ ]:
# import DFF ensemble matrix 
dff_ens_path = results_dir / Path(r"dff_ensemble_detection_21-Nov-2025_5000 shuffles\dff_postoutcome_python_ensembles_5000_shuff_21-Nov-2025_.parquet")
dff_ens = pd.read_parquet(dff_ens_path)
if 'data_type' not in dff_ens.columns:
    dff_ens['data_type'] = 'dff'
print(dff_ens.info())
#get ensemble info from canon df
dff_ens.tail(3)

In [ ]:
#import last spikes ensemble matrix for comparison
spikes_ens_path = results_dir / Path(r"spikes_ensemble_detection_20-Nov-2025_5000 shuffles\spikes_postoutcome_python_ensembles_5000_shuff_20-Nov-2025_.parquet")
spikes_ens = pd.read_parquet(spikes_ens_path)
if 'data_type' not in spikes_ens.columns:
    spikes_ens['data_type'] = 'spikes'
print(spikes_ens.info())
spikes_ens.tail(3)

In [ ]:
## pivot both enrichment vectors, then merge
spikes_ens_matrix = spikes_ens.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = 'enriched', fill_value = None).astype(float)
dff_ens_matrix = dff_ens.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = 'enriched', fill_value = None).astype(float)
compare_spike_dff_ens = spikes_ens_matrix.merge(dff_ens_matrix, on = 'neuron_id', suffixes=('_spikes', '_dff'))
print(compare_spike_dff_ens.info())
compare_spike_dff_ens.head(3)

In [ ]:
df = compare_spike_dff_ens
correlations = {stage: df[stage+"_spikes"].corr(df[stage + '_dff']) for stage in stage_names}
corr_df = pd.DataFrame.from_dict(correlations, orient='index', columns=['Correlation'])
print(f" Correlation of enrichment vectors for DFF data vs SPIKE data: \n{corr_df}")

In [ ]:
## pivot both p-value matrices, then merge
spikes_ens_pval = spikes_ens.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = 'percent_shuffle >= real', fill_value = False).astype(float)
dff_ens_pval = dff_ens.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = 'percent_shuffle >= real', fill_value = False).astype(float)
compare_spike_dff_p = spikes_ens_pval.merge(dff_ens_pval, on = 'neuron_id', suffixes=('_spikes', '_dff'))
print(compare_spike_dff_p.info())
compare_spike_dff_p.head(3)

In [ ]:
corr_df = get_intercol_corrs(compare_spike_dff_p, stage_names, suffix1="_spikes", suffix2="_dff")
print(f" Correlation of enrichment vectors for DFF data vs SPIKE data: \n{corr_df}")

#### Threshold time-series df to remove 0 

In [ ]:
from preprocess_data import hyper_param_dict, bin_rotate_timeseries, get_numeric_cols_timeseries, get_unit_mean_timeseries_by_phase, drop_end_bins_of_trials, drop_start_bins_of_trials, add_enriched_in_curr_phase_col, get_subject_stage_info_df
from helper_functions import annotate_csv, run_min_max_norm_on_timeseries


In [ ]:
#for concordance with previous code, need to label the stage col as 'task_stage_vec
stage_col = 'task_phase_vec'

# ## main time-series df creation function for datasets processed in python
# def create_subject_trial_tseries_df(input_df: pd.DataFrame,
#                                     ens_matrix: pd.DataFrame,
#                                     window_to_bin = 5,
#                                     n_sec_to_rotate = 0,
#                                     cell_col = 'cell',
#                                     cols_to_drop= None,
#                                     stage_col = 'task_phase_vec',
#                                     stage_names = None,
#                                     ) -> pd.DataFrame:
#     """ Create trial-cell time-series DataFrame for one subject, with ensemble info joined. For python native created dfs
#     Inputs:
#     raster_df : DataFrame with one row per frame
#     """
#     #default args
#     if cols_to_drop is None:
#         cols_to_drop = ['threshold_with_shuffle','peak_dff_threshold_percentile','drop_low_value_peak_events','cutoff_filter','peak_event_cutoff_percentile']   
#     if stage_names is None:
#         stage_names = ['Early_IA_Error', 'Early_IA_Correct', 'Late_IA', 'Early_RS_Error', 'Early_RS_Correct','Late_RS']
        
#     # extract trial windows from df, removing excess frames
#     trim_df = extract_trial_windows(input_df,pre_frames=config['timeseries']['pre_frames'], post_frames= config['timeseries']['post_frames'])
#     #reshape df into long format: one row per trial-cell, columns = time-series frames
#     outcome_post = pivot_trial_windows(trim_df).rename(columns={'task_stage': stage_col})
#     outcome_post['neuron_id'] = outcome_post['subject_name'] + '-' + outcome_post[cell_col].str.replace('cell_','') 
#     outcome_post =outcome_post.drop([x for x in cols_to_drop if x in outcome_post], axis = 1).dropna(subset = [stage_col])
#     annotate_csv(outcome_post, 'subject_name')

#     ## begin preprocess- bin and rotate timeseries 
#     outcome_post = bin_rotate_timeseries(outcome_post, window_size = window_to_bin, rotate_by = n_sec_to_rotate)
#     outcome_post = outcome_post.join(ens_matrix, on='neuron_id', how='left', rsuffix='_ens') #join dff timeseries with ensemble info
#     #drop time-series not belonging to any task stage , aka the 0 task stage 
#     outcome_post['any_enrichment']= outcome_post[stage_names].sum(axis = 1)
#     outcome_post = outcome_post[outcome_post[stage_col].isin(stage_names)]
#     trial_tseries_df_raw = add_enriched_in_curr_phase_col(outcome_post, stage_col)
#     return trial_tseries_df_raw
# # 


def normalize_tseries_df(trial_tseries_df_raw: pd.DataFrame,
                          n_end_timebins_to_drop = config['timeseries']['n_end_timebins_to_drop'], #default = 0,
                          n_start_timebins_to_drop = config['timeseries']['n_start_timebins_to_drop'], #default = 8,
                          name_col: str = 'subject_name',
                          neuron_id_col: str = 'neuron_id',
                          ) -> pd.DataFrame:
    """ Normalize trial time-series DataFrame based on normalization type.
    Inputs:
    trial_tseries_df_raw : DataFrame with raw time-series data (ALREADY BINNED)
    hyper_param_dict : dictionary of hyperparameters
    outcome_post : DataFrame with outcome post data

    Returns: Normalized trial time-series DataFrame
    """
    #get normalization info for curr df
    normed = trial_tseries_df_raw['normalized'].unique()[0]
    if normed == 'none':
        use_min_max_norm = True 
    else:
        use_min_max_norm = False
    print(f" Data Normalization type: {normed} | Applying min-max norm? {use_min_max_norm} ")

    numeric_col = get_numeric_cols_timeseries(trial_tseries_df_raw, " to ") 
    #is min-max norm run BEFORE or AFTER bin dropping? 
    trial_tseries_df_norm = run_min_max_norm_on_timeseries(use_min_max_norm, trial_tseries_df_raw, [name_col, neuron_id_col], numeric_col, 'max_trial_val') #min_max_norm = True

    #get mean of each trial- IS this run BEFORE or AFTER norm? 
    trial_tseries_df_norm['mean_rate'] =trial_tseries_df_norm[[c for c in numeric_col if '-' not in c]].mean(axis = 1) #mean rate is post outcome only
    trial_tseries_df_norm['active_in_trial'] =trial_tseries_df_norm['mean_rate']>0

    #OPTIONAL- drop N bins from start and M bins from end of time-series
    #first, check if you're pre-truncated the data (e.g. already only sliced 3 seconds before outcome)
    neg_timebins = [c for c in numeric_col if "-" in c]
    if len(neg_timebins) <4*3+1: #if you only have 1 second or so of pre-outcome data, you've already sliced 
        n_start_timebins_to_drop = 0
    trial_tseries_df_norm = drop_end_bins_of_trials(trial_tseries_df_norm,numeric_col,  n_end_timebins_to_drop = n_end_timebins_to_drop )
    trial_tseries_df_norm = drop_start_bins_of_trials(trial_tseries_df_norm,numeric_col,  n_start_timebins_to_drop =n_start_timebins_to_drop)
    return trial_tseries_df_norm

def normalize_all_subject_tseries_dfs(raster_base_dfs: Dict[str, pd.DataFrame],
                                      config = config, 
                                      n_end_timebins_to_drop = 0,
                                      n_start_timebins_to_drop = 8,
                                      name_col: str = 'subject_name',
                                      neuron_id_col: str = 'neuron_id',
                                      ) -> pd.DataFrame:
    """ Normalize all subject trial time-series DataFrames.
    Inputs:     raster_base_dfs : Dict of subject_name to raw full recording matrix DataFrame
    optional settings for time-series normalization that equal defaults, so left un input

    Returns: concat. dataframe of subject_name to normalized trial time-series DataFrame
    """

    raster_timeseries =[]
    for name, raster_df in raster_base_dfs.items():
        print(f" Reshaping tseries of {name}: {raster_df.shape}")    
        subj_trial_tseries_df_raw= create_subject_trial_tseries_df(raster_df, dff_ens_matrix, config = config)
        subj_trial_tseries_df_norm= normalize_tseries_df(subj_trial_tseries_df_raw)
        raster_timeseries.append(subj_trial_tseries_df_norm)
    trial_tseries_df_norm = pd.concat(raster_timeseries)
    return trial_tseries_df_norm

In [ ]:
trial_tseries_df_norm = normalize_all_subject_tseries_dfs(raster_dataframes, config = config)
trial_tseries_df_norm


In [ ]:

# get mean time-series value
numeric_col = get_numeric_cols_timeseries(trial_tseries_df_norm, " to ")  #update post drop
unit_mean_tseries =  get_unit_mean_timeseries_by_phase(trial_tseries_df_norm,['trial_num'], ['subject_name', 'neuron_id','geno_day', stage_col], numeric_col) 
unit_mean_tseries


In [ ]:
n_total_units = unit_mean_tseries.groupby(by = 'geno_day')['neuron_id'].nunique()
print(n_total_units)
print(n_total_units.sum())
#save filename with data type and date info
save_name = results_dir / Path(f"python-made_trial_timeseries_{data_type_used}_{datetime.now().strftime("%Y-%m-%d %H")}.parquet")
print(f"saving file as {save_name}")
trial_tseries_df_norm.to_parquet(save_name)


#### Import canon ensemble + tseries to compare umap

In [ ]:
# import canonical ensemble matrix previously created in MATLAB, used for paper #find old file- 
canonical_file = data_dir / Path(r"Dlx56_Normalized Trial Calcium Timeseries_20_Jun_2025.parquet")
canonical_data = pd.read_parquet(canonical_file)
print(canonical_data.info())

canonical_data

In [ ]:
#get canon mean unit timeseries
numeric_col_canon = get_numeric_cols_timeseries(canonical_data, " to ")  #update post drop\
canon_mean_unit_tseries  =  get_unit_mean_timeseries_by_phase(canonical_data,['trial_num'], ['name', 'neuron_ID','geno_day', stage_col], numeric_col) 
print(canon_mean_unit_tseries.groupby(by = 'geno_day')['unique_ID'].nunique())
print(canon_mean_unit_tseries.info())
canon_mean_unit_tseries


In [ ]:
print(f"numerical cols of CANON timeseries are: {numeric_col_canon}")
print(f"numerical cols of new python timeseries are: {numeric_col}")

In [ ]:
neg_timebins = [c for c in numeric_col if "-" in c]
print(len(neg_timebins)) 
neg_timebins 

#### compare time-series of CANON and NEW PYTHON

#### UMAP on enrichment matrices

In [ ]:
# UMAP analysis on ensemble matrices
import umap
from sns_plotting_config import * #import dicts containing default plot params


In [ ]:
# # Determine first active stage for each neuron
def get_first_active_stage(row, stage_order,  non_enriched_label = 'Never'):
    """Find the first stage where neuron is enriched (value = 1)"""
    for stage in stage_order:
        if stage in row.index and row[stage] == 1:
            return stage
    return non_enriched_label

def run_dimensionality_reduction(ensemble_matrix, method='umap', metric='cosine', random_state=42, **method_kwargs):
    """Run UMAP or t-SNE on ensemble matrix. Returns (df, model)."""
    from sklearn.manifold import TSNE
    import umap
    import pandas as pd
    complete_data = ensemble_matrix.dropna()
    print(f"Shape: {ensemble_matrix.shape} → {complete_data.shape}")
    
    if method.lower() == 'umap':
        model = umap.UMAP(n_neighbors=method_kwargs.get('n_neighbors', 15), min_dist=method_kwargs.get('min_dist', 0.1), 
                          metric=metric, random_state=random_state)
    elif method.lower() == 'tsne':
        model = TSNE(perplexity=method_kwargs.get('perplexity', 30), learning_rate=method_kwargs.get('learning_rate', 200), n_iter=method_kwargs.get('n_iter', 1000),
            metric=metric, random_state=random_state)
    else:
        raise ValueError(f"method must be 'umap' or 'tsne', got '{method}'")
    
    embedding = model.fit_transform(complete_data)
    results_df = pd.DataFrame({'dim1': embedding[:, 0], 'dim2': embedding[:, 1], 'method': method.upper()}, index=complete_data.index)
    return results_df, model

In [ ]:
#set umap params
value_to_project = 'mean'
metric = 'cosine'

#package spikes + dff for umap processing 
spike_umap_input = spikes_ens.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = value_to_project, fill_value = False).drop(columns = 'truncated_post_outcome').astype(float)
dff_umap_input = dff_ens.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = value_to_project, fill_value = False).drop(columns = 'truncated_post_outcome').astype(float)

# Run UMAP on both spike and DFF ensemble matrices
spikes_df, umap_spikes_model = run_dimensionality_reduction( spike_umap_input, method = 'umap', n_neighbors=15, min_dist=0.2, metric=metric,random_state=42)
dff_df, umap_dff_model = run_dimensionality_reduction(dff_umap_input, method = 'umap', n_neighbors=15, min_dist=0.2, metric=metric, random_state=42)

#join umap dfs with ensemble enrichment info
spike_cell_enrich = spikes_ens.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = 'enriched', fill_value = False).astype(float).apply(get_first_active_stage, axis=1, stage_order=stage_names)
spikes_df = spikes_df.merge(spike_cell_enrich.rename('first_active_stage'), left_index=True, right_index=True)

dff_cell_enrich = dff_ens.pivot_table(index = ['neuron_id'], columns = 'task_stage', values = 'enriched', fill_value = False).astype(float).apply(get_first_active_stage, axis=1, stage_order=stage_names)
dff_df = dff_df.merge(dff_cell_enrich.rename('first_active_stage'), left_index=True, right_index=True)


In [ ]:
spike_umap_input

In [ ]:
#new- 12/3/25- import canon timeseries + taco
#make pivot of mean activity of cell over its timeseries 
canon_spike_umap_input = canon_mean_unit_tseries.pivot_table(index = ['unique_ID'], columns = 'task_phase_vec', values = 'mean_rate', fill_value = False)
canon_spike_umap_input

In [ ]:
spike_mean_concat = pd.merge(canon_spike_umap_input, spike_umap_input.astype(float), 
         how = 'inner', left_index = True, right_index = True, 
         suffixes=  ('_canon','_new'))
spike_mean_concat.head()

In [ ]:
diff_cols = {}
for s in stage_names:
    diff_cols[s] = (spike_mean_concat[s+"_canon"]- spike_mean_concat[s+"_new"]).astype(float)
    non_zero_rows = diff_cols[s] != 0
    # diff_cols[s] [non_zero_rows]= 100*diff_cols[s][non_zero_rows] / spike_mean_concat.loc[non_zero_rows, s+"_canon"].astype(float)
canon_less_new_mean= pd.DataFrame(diff_cols)
canon_less_new_mean 

In [ ]:
canon_less_new_mean.describe()

In [ ]:
sns.heatmap(canon_less_new_mean.astype(float))

#### slice changed cells to further investigate

In [ ]:
canon_cell_diff_bool = canon_less_new_mean != 0
canon_cell_diff_bool

#### Plot low-D projections: UMAP

In [ ]:
canon_spikes_df, canon_umap_spikes_model = run_dimensionality_reduction( canon_spike_umap_input, method = 'umap', n_neighbors=15, min_dist=0.2, metric=metric,random_state=42)

canon_spike_cell_enrich = canon_mean_unit_tseries.groupby(by = 'unique_ID')[stage_names].first().astype(float).apply(get_first_active_stage, axis=1, stage_order=stage_names)
canon_spikes_df = canon_spikes_df.merge(canon_spike_cell_enrich.rename('first_active_stage'), left_index=True, right_index=True)
canon_spikes_df

In [ ]:
## PLOTTING- note that paper tSNE is projection of mean activity in stage, colored by ensemble identity
fig, axes = plt.subplots(1, 3, figsize=(7, 2), layout='constrained')
palette = [stage_palette_dict[x] for x in stage_names] + ['lightgray']
hue_col = 'first_active_stage'

# Spikes
sns.scatterplot(ax=axes[0], data=spikes_df, x='dim1', y='dim2', hue=hue_col,     palette=palette, hue_order=stage_names , s=10, alpha=0.5, edgecolor='k',linewidth=0.5, legend='full')
axes[0].set_title(f'Spike data: Colored by First Active Stage')
axes[0].grid(True, alpha=0.3)
# Spikes- CANON
sns.scatterplot(ax=axes[2], data=canon_spikes_df, x='dim1', y='dim2', hue=hue_col,     palette=palette, hue_order=stage_names , s=10, alpha=0.5, edgecolor='k',linewidth=0.5, legend='full')
axes[2].set_title(f'CANON Spike data: Colored by First Active Stage')
axes[2].grid(True, alpha=0.3)


# DFF
sns.scatterplot(ax=axes[1], data=dff_df, x='dim1', y='dim2', hue=hue_col, palette=palette, hue_order=stage_names , s=10, alpha=0.5, edgecolor='k',  linewidth=0.5, legend='full')
axes[1].set_title(f'DFF data: Hue is First Active Stage')
axes[1].grid(True, alpha=0.3)

#Final check- save manifest of config

In [ ]:
subjects= unit_mean_tseries['subject_name'].unique().tolist()
enrichment_output_path = "None"
timeseries_output_path= "None"

In [ ]:
# Cell: Save run manifest
runtime = time.time() - START_TIME

manifest = create_run_manifest(
    config=config,
    output_dir=run_output_dir,
    run_id=RUN_ID,
    inputs={'source_data': str(config['data']['source_dataset_location']),'n_subjects': len(subjects),},
    outputs={'shuffle_parquets': str(run_output_dir),
        'n_files': len(list(run_output_dir.glob('*.parquet'))),
        'enrichment_results': str(enrichment_output_path),
        'timeseries_results': str(timeseries_output_path),
    },
    performance={'runtime_seconds': int(runtime),'runtime_human': f"{runtime/3600:.1f} hours",'n_cores_used': config['shuffles']['n_cores'],},
    status={'success': True,'errors': [],'warnings': [],}
)

# Update central run log
run_db = update_run_database(manifest, db_path=results_dir / 'run_log.csv')
print(f"Run complete: {RUN_ID}")
print(f"Runtime: {runtime/3600:.1f} hours")
print(f"Outputs: {run_output_dir}")
print(f"Manifest: {run_output_dir / f'run_manifest_{RUN_ID}.yaml'}")

#### TEMP- COMPARE TIMESERIES

In [ ]:
"""
Fixed analysis - check for proper neuron ID mapping
"""

import pandas as pd
import numpy as np
from pathlib import Path

# Load data

# canon_path = data_dir / "Dlx56_Normalized Trial Calcium Timeseries_20_Jun_2025.parquet"
canon_path = data_dir / "post_outcome_main_datasets_neurons_trial activity timeseries data_10-Dec-2024_timeseries.csv"
new_path = results_dir / save_name

print("="*80)
print("LOADING DATA")
print("="*80)
canon_df = pd.read_parquet(canon_path)
# new_df = pd.read_parquet(new_path)
# Re-load the newly generated data
new_df = trial_tseries_df_norm  # Or however you reference it
print(f"CANON shape: {canon_df.shape}")
print(f"NEW shape: {new_df.shape}")

# Check neuron ID columns
print("\n" + "="*80)
print("NEURON ID INVESTIGATION")
print("="*80)

print("\nCANON has these ID-related columns:")
id_cols_canon = [c for c in canon_df.columns if 'id' in c.lower() or 'neuron' in c.lower() or 'unique' in c.lower()]
print(f"  {id_cols_canon}")

for col in id_cols_canon:
    print(f"\n{col}:")
    print(f"  dtype: {canon_df[col].dtype}")
    print(f"  unique count: {canon_df[col].nunique()}")
    print(f"  sample values: {canon_df[col].unique()[:5]}")

print("\nNEW has these ID-related columns:")
id_cols_new = [c for c in new_df.columns if 'id' in c.lower() or 'neuron' in c.lower() or 'unique' in c.lower()]
print(f"  {id_cols_new}")

for col in id_cols_new:
    print(f"\n{col}:")
    print(f"  dtype: {new_df[col].dtype}")
    print(f"  unique count: {new_df[col].nunique()}")
    print(f"  sample values: {new_df[col].unique()[:5]}")

# Check if unique_ID in CANON matches neuron_id in NEW
print("\n" + "="*80)
print("CHECKING FOR MATCHES")
print("="*80)

if 'unique_ID' in canon_df.columns:
    canon_unique_ids = set(canon_df['unique_ID'].astype(str).unique())
    new_neuron_ids = set(new_df['neuron_id'].astype(str).unique())
    
    overlap = canon_unique_ids & new_neuron_ids
    
    print(f"\nCANON unique_ID count: {len(canon_unique_ids)}")
    print(f"NEW neuron_id count: {len(new_neuron_ids)}")
    print(f"Overlap: {len(overlap)}")
    print(f"Only in CANON: {len(canon_unique_ids - new_neuron_ids)}")
    print(f"Only in NEW: {len(new_neuron_ids - canon_unique_ids)}")
    
    if len(overlap) > 0:
        print(f"\nGOOD NEWS! Found {len(overlap)} matching neurons")
        print(f"Sample matches: {list(overlap)[:10]}")
    else:
        print("\nNO MATCHES FOUND between unique_ID and neuron_id")
        print("\nSample CANON unique_ID values:")
        print(f"  {list(canon_unique_ids)[:10]}")
        print("\nSample NEW neuron_id values:")
        print(f"  {list(new_neuron_ids)[:10]}")

# Check time-series columns
print("\n" + "="*80)
print("TIME-SERIES COLUMNS")
print("="*80)

canon_ts_cols = [col for col in canon_df.columns if 's to' in col]
new_ts_cols = [col for col in new_df.columns if 's to' in col]

print(f"\nCANON has {len(canon_ts_cols)} time-series columns")
print(f"NEW has {len(new_ts_cols)} time-series columns")
print(f"\nFirst 5 CANON: {canon_ts_cols[:5]}")
print(f"First 5 NEW: {new_ts_cols[:5]}")
print(f"\nTime-series columns match: {canon_ts_cols == new_ts_cols}")

In [ ]:
"""
CORRECTED Analysis - comparing matching neurons between CANON and NEW
"""

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# Load data
# canon_path = data_dir / "post_outcome_main_datasets_neurons_trial activity timeseries data_10-Dec-2024_timeseries.csv"
# new_path = results_dir / "python-made_trial_timeseries_raster_2025-12-04 18.parquet"

print("="*80)
print("LOADING DATA")
print("="*80)
canon_df = pd.read_parquet(canon_path)
# Re-load the newly generated data
new_df = trial_tseries_df_norm  # Or however you reference it
# new_df = pd.read_parquet(new_path)
print(f"CANON shape: {canon_df.shape}")
print(f"NEW shape: {new_df.shape}")

# Get time-series columns
canon_ts_cols = [col for col in canon_df.columns if 's to' in col]
new_ts_cols = [col for col in new_df.columns if 's to' in col]

print(f"\nTime-series columns: {len(canon_ts_cols)} (both files match)")

# ==============================================================================
# CALCULATE MEAN ACTIVITY BY TASK STAGE FOR EACH NEURON
# ==============================================================================
print("\n" + "="*80)
print("CALCULATING MEAN ACTIVITY BY TASK STAGE")
print("="*80)

# CANON - use unique_ID (not neuron_ID!)
canon_mean_by_stage = (canon_df.groupby(['unique_ID', 'task_phase_vec'])[canon_ts_cols]
                       .mean()
                       .mean(axis=1)  # Mean across all timepoints
                       .reset_index(name='mean_activity'))

print(f"\nCANON mean activity shape: {canon_mean_by_stage.shape}")
print(f"Unique neurons in CANON: {canon_mean_by_stage['unique_ID'].nunique()}")
print(f"Sample:\n{canon_mean_by_stage.head(10)}")

# NEW
new_mean_by_stage = (new_df.groupby(['neuron_id', 'task_phase_vec'])[new_ts_cols]
                     .mean()
                     .mean(axis=1)  # Mean across all timepoints
                     .reset_index(name='mean_activity'))

print(f"\nNEW mean activity shape: {new_mean_by_stage.shape}")
print(f"Unique neurons in NEW: {new_mean_by_stage['neuron_id'].nunique()}")
print(f"Sample:\n{new_mean_by_stage.head(10)}")

# ==============================================================================
# MERGE AND COMPARE
# ==============================================================================
print("\n" + "="*80)
print("MERGING AND COMPARING MEAN ACTIVITIES")
print("="*80)

# Rename for merging
canon_mean_renamed = canon_mean_by_stage.rename(columns={
    'unique_ID': 'neuron_id',
    'task_phase_vec': 'task_stage',
    'mean_activity': 'canon_mean'
})

new_mean_renamed = new_mean_by_stage.rename(columns={
    'task_phase_vec': 'task_stage',
    'mean_activity': 'new_mean'
})

# Merge
comparison = canon_mean_renamed.merge(
    new_mean_renamed,
    on=['neuron_id', 'task_stage'],
    how='outer',
    indicator=True
)

print(f"\nMerge result shape: {comparison.shape}")
print(f"\nMerge indicator counts:")
print(comparison['_merge'].value_counts())

# Calculate difference
comparison['difference'] = comparison['new_mean'] - comparison['canon_mean']
comparison['abs_difference'] = comparison['difference'].abs()
comparison['pct_difference'] = (comparison['difference'] / comparison['canon_mean']) * 100

# ==============================================================================
# ANALYZE DIFFERENCES
# ==============================================================================
print("\n" + "="*80)
print("DIFFERENCE STATISTICS")
print("="*80)

# Filter to only matching records
matched = comparison[comparison['_merge'] == 'both'].copy()
print(f"\nMatched neuron-stage pairs: {len(matched)}")
print(f"Unique neurons in matched pairs: {matched['neuron_id'].nunique()}")

print(f"\nDifference statistics:")
print(matched[['difference', 'abs_difference', 'pct_difference']].describe())

# Find cells with largest differences
print(f"\n\nTop 20 neuron-stage pairs with largest absolute differences:")
top_diffs = matched.nlargest(20, 'abs_difference')[
    ['neuron_id', 'task_stage', 'canon_mean', 'new_mean', 'difference', 'pct_difference']
]
print(top_diffs.to_string())

# Cells with any difference
cells_with_diff = matched[matched['abs_difference'] > 1e-10]['neuron_id'].unique()
print(f"\n\nCells with ANY difference (>1e-10): {len(cells_with_diff)}")
print(f"Total unique cells in comparison: {matched['neuron_id'].nunique()}")
print(f"Percentage with differences: {len(cells_with_diff)/matched['neuron_id'].nunique()*100:.1f}%")

# Cells with significant difference (>1% or >0.01 absolute)
cells_with_sig_diff = matched[
    (matched['abs_difference'] > 0.01) | (matched['pct_difference'].abs() > 1)
]['neuron_id'].unique()
print(f"\nCells with SIGNIFICANT difference (>1% or >0.01 abs): {len(cells_with_sig_diff)}")
print(f"Percentage: {len(cells_with_sig_diff)/matched['neuron_id'].nunique()*100:.1f}%")

# By stage analysis
print(f"\n\nDifferences by task stage:")
stage_stats = matched.groupby('task_stage').agg({
    'difference': ['mean', 'std', 'min', 'max'],
    'abs_difference': ['mean', 'max'],
    'neuron_id': 'count'
})
print(stage_stats)

# Check which neurons are missing
print("\n" + "="*80)
print("MISSING NEURONS")
print("="*80)

only_canon = comparison[comparison['_merge'] == 'left_only']['neuron_id'].unique()
only_new = comparison[comparison['_merge'] == 'right_only']['neuron_id'].unique()

print(f"\nNeurons only in CANON: {len(only_canon)}")
if len(only_canon) <= 20:
    print(f"  {sorted(only_canon)}")

print(f"\nNeurons only in NEW: {len(only_new)}")
if len(only_new) <= 20:
    print(f"  {sorted(only_new)}")

# ==============================================================================
# VISUALIZATION
# ==============================================================================
print("\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Scatter plot: CANON vs NEW
ax = axes[0, 0]
ax.scatter(matched['canon_mean'], matched['new_mean'], alpha=0.3, s=1)
ax.plot([matched['canon_mean'].min(), matched['canon_mean'].max()],
        [matched['canon_mean'].min(), matched['canon_mean'].max()],
        'r--', lw=2, label='y=x')
ax.set_xlabel('CANON Mean Activity')
ax.set_ylabel('NEW Mean Activity')
ax.set_title('CANON vs NEW Mean Activity\n(Each point = one neuron-stage pair)')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Histogram of differences
ax = axes[0, 1]
ax.hist(matched['difference'], bins=100, edgecolor='black')
ax.set_xlabel('Difference (NEW - CANON)')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of Differences\nMean={matched["difference"].mean():.6f}, Std={matched["difference"].std():.6f}')
ax.axvline(0, color='red', linestyle='--', lw=2)
ax.grid(True, alpha=0.3)

# 3. Absolute difference by stage
ax = axes[1, 0]
stage_groups = matched.groupby('task_stage')['abs_difference']
stage_names = list(stage_groups.groups.keys())
stage_values = [stage_groups.get_group(s).values for s in stage_names]
ax.boxplot(stage_values, labels=stage_names)
ax.set_ylabel('Absolute Difference')
ax.set_xlabel('Task Stage')
ax.set_title('Absolute Difference Distribution by Task Stage')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

# 4. Percentage difference distribution
ax = axes[1, 1]
# Filter out infinite and very large values for visualization
pct_diff_filtered = matched['pct_difference'][
    (matched['pct_difference'].abs() < 100) &
    (matched['pct_difference'].notna())
]
ax.hist(pct_diff_filtered, bins=100, edgecolor='black')
ax.set_xlabel('Percentage Difference (%)')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of % Differences\n(Filtered to ±100%)')
ax.axvline(0, color='red', linestyle='--', lw=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'timeseries_comparison.png', dpi=150, bbox_inches='tight')
print(f"\nSaved visualization to: {results_dir / 'timeseries_comparison.png'}")
plt.show()

# Save comparison data
comparison_path = results_dir / 'timeseries_comparison_detailed.parquet'
matched.to_parquet(comparison_path)
print(f"\nSaved detailed comparison to: {comparison_path}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print(f"\nSUMMARY:")
print(f"  Neurons in CANON: 4776")
print(f"  Neurons in NEW: 4814")
print(f"  Neurons in both: 4676")
print(f"  Matched neuron-stage pairs: {len(matched)}")
print(f"  Pairs with ANY difference: {len(cells_with_diff)}")
print(f"  Pairs with SIGNIFICANT difference: {len(cells_with_sig_diff)}")

In [ ]:
# For one neuron-stage pair with large difference:
test_neuron = '13_4_WT_RS2-42'
test_stage = 'Early_IA_Error'

canon_trials = canon_df[(canon_df['unique_ID'] == test_neuron) & 
                        (canon_df['task_phase_vec'] == test_stage)]['trial_num'].values
new_trials = new_df[(new_df['neuron_id'] == test_neuron) & 
                    (new_df['task_phase_vec'] == test_stage)]['trial_num'].values

print(f"CANON trial_nums: {sorted(canon_trials)}")
print(f"NEW trial_nums: {sorted(new_trials)}")
print(f"Same trials? {set(canon_trials) == set(new_trials)}")

# Check if values differ for a specific trial
if len(set(canon_trials) & set(new_trials)) > 0:
    test_trial = list(set(canon_trials) & set(new_trials))[0]
    
    canon_vals = canon_df[(canon_df['unique_ID'] == test_neuron) & 
                          (canon_df['task_phase_vec'] == test_stage) &
                          (canon_df['trial_num'] == test_trial)][canon_ts_cols].values[0]
    
    new_vals = new_df[(new_df['neuron_id'] == test_neuron) & 
                      (new_df['task_phase_vec'] == test_stage) &
                      (new_df['trial_num'] == test_trial)][new_ts_cols].values[0]
    
    print(f"\nTrial {test_trial} values match? {np.allclose(canon_vals, new_vals, equal_nan=True)}")
    print(f"Max difference: {np.nanmax(np.abs(canon_vals - new_vals))}")

In [ ]:
canon_df

In [ ]:
canon_ts_cols

In [ ]:
test_neuron = '13_4_WT_RS2-42'
test_stage = 'Early_IA_Error'
test_trial = 4

# Get the time-series values for this specific trial
canon_row = canon_df[(canon_df['unique_ID'] == test_neuron) & 
                      (canon_df['task_phase_vec'] == test_stage) &
                      (canon_df['trial_num'] == test_trial)]

new_row = new_df[(new_df['neuron_id'] == test_neuron) & 
                  (new_df['task_phase_vec'] == test_stage) &
                  (new_df['trial_num'] == test_trial)]

print("CANON values (first 20 bins size):")
print(canon_row[canon_ts_cols[:20]].values[0].size)

print("\nNEW values (first 20 bins size):")
print(new_row[new_ts_cols[:20]].values[0].size)



print("CANON values (first 20 bins):")
print(canon_row[canon_ts_cols[:20]].values[0])

print("\nNEW values (first 20 bins):")
print(new_row[new_ts_cols[:20]].values[0])

print("\n\nCANON mean across all bins:", canon_row[canon_ts_cols].values[0].mean())
print("NEW mean across all bins:", new_row[new_ts_cols].values[0].mean())

print("\n\nCANON max_trial_val:", canon_row['max_trial_val'].values[0] if 'max_trial_val' in canon_row.columns else "N/A")
print("NEW max_trial_val:", new_row['max_trial_val'].values[0] if 'max_trial_val' in new_row.columns else "N/A")

# Check how many bins are non-zero
print("\n\nCANON non-zero bins:", (canon_row[canon_ts_cols].values[0] > 0).sum())
print("NEW non-zero bins:", (new_row[new_ts_cols].values[0] > 0).sum())

In [ ]:
# For this neuron (13_4_WT_RS2-42) in trial 4:
# What are the RAW frame values (before binning) in the pre-outcome section?

# In the raster DataFrame (before extract_trial_windows), for trial 4:
test_subject = '13_4_WT_RS2'  # Adjust to match your subject naming
test_neuron_idx = 42  # Adjust if needed

# Get the raw raster for this subject
subject_df = raster_dataframes[test_subject]  # or whatever your dict is called

# Filter to trial 4, pre_outcome section
trial_4_pre = subject_df[(subject_df['trial_num'] == 4) & 
                          (subject_df['trial_section'] == 'pre_outcome')]

# Show the activity values for this neuron
cell_col = 'cell_42'  # or however it's named
print(f"Number of pre-outcome frames in trial 4: {len(trial_4_pre)}")
print(f"Cell activity in these frames:")
print(trial_4_pre[cell_col].values)

In [ ]:
trial_4_pre

In [ ]:
print("Frames 861-960 values:")
print(trial_4_pre[cell_col].values[860:960])  # 0-indexed

In [ ]:
# Load the original MATLAB CSV that was used to create CANON
# This is the CSV exported from MATLAB's return_section_activity_timeseries.m
# and fed into the Colab notebook's get_normed_trial_tseries function

import pandas as pd
import numpy as np
from pathlib import Path

# Path to the MATLAB-exported CSV
# Adjust this path based on where the file is located
# matlab_csv_path = Path(r"G:\My Drive\Colab Notebooks\Sohal Lab Datasets\time-series including WT CLNZ dataset\11-Dec-2024_event_fixed_stage_labels Dataset Source\post_outcome_main_datasets_neurons_trial activity timeseries data_10-Dec-2024_timeseries.csv")

# Or if it's local:
matlab_csv_path = data_dir / Path(r"post_outcome_WT_CLNZ_neurons_trial activity timeseries data_18-Nov-2025_timeseries.csv")

print(f"Loading MATLAB CSV from: {matlab_csv_path}")
print(f"File exists: {matlab_csv_path.exists()}")

if matlab_csv_path.exists():
    matlab_csv = pd.read_csv(matlab_csv_path, low_memory=False)
    print(f"\nMATLAB CSV shape: {matlab_csv.shape}")
    print(f"Columns (first 20): {matlab_csv.columns.tolist()[:20]}")
    
    # Find the same neuron and trial
    # In MATLAB CSV, neuron_ID is just the cell number (42), not the full string
    # We need to filter by name (subject) and neuron_ID
    test_subject = '13_4_WT_RS2'
    test_neuron_id = 42
    test_trial = 4
    
    # Filter to this specific neuron-trial combination
    matlab_row = matlab_csv[(matlab_csv['name'] == test_subject) & 
                            (matlab_csv['neuron_ID'] == test_neuron_id) &
                            (matlab_csv['trial_num'] == test_trial)]
    
    if len(matlab_row) == 0:
        print(f"\nTrying alternate subject name matching...")
        print(f"Unique subject names in MATLAB CSV: {matlab_csv['name'].unique()[:10]}")
    else:
        print(f"\nFound {len(matlab_row)} matching row(s) in MATLAB CSV")
        
        # Get frame columns (they should be labeled like 'f_1', 'f_2', etc. and '-f_1', '-f_2', etc.)
        frame_cols = [col for col in matlab_csv.columns if col.startswith(('f_', '-f_'))]
        pre_frame_cols = sorted([col for col in frame_cols if col.startswith('-f_')], 
                                key=lambda x: int(x.split('_')[1]), reverse=True)
        post_frame_cols = sorted([col for col in frame_cols if col.startswith('f_') and not col.startswith('-f_')],
                                 key=lambda x: int(x.split('_')[1]))
        
        print(f"\nPre-outcome frame columns: {len(pre_frame_cols)}")
        print(f"  First 5: {pre_frame_cols[:5]}")
        print(f"  Last 5: {pre_frame_cols[-5:]}")
        
        print(f"\nPost-outcome frame columns: {len(post_frame_cols)}")
        print(f"  First 5: {post_frame_cols[:5]}")
        print(f"  Last 5: {post_frame_cols[-5:]}")
        
        # Extract the pre-outcome frame values
        matlab_pre_values = matlab_row[pre_frame_cols].values[0]
        
        print(f"\n{'='*80}")
        print("MATLAB CSV PRE-OUTCOME VALUES (frames before binning):")
        print(f"{'='*80}")
        print(f"Number of pre-frames: {len(matlab_pre_values)}")
        print(f"Values: {matlab_pre_values}")
        print(f"Non-zero count: {(matlab_pre_values > 0).sum()}")
        print(f"Mean: {matlab_pre_values.mean():.4f}")
        
        # Compare to NEW pipeline's raw data
        # We already know NEW has frames 861-960 from the 960-frame pre-outcome
        # And frames 901-960 were extracted (last 60)
        new_extracted_frames = trial_4_pre[cell_col].values[900:960]  # 0-indexed, so 901-960 is [900:960]
        
        print(f"\n{'='*80}")
        print("NEW PIPELINE PRE-OUTCOME VALUES (frames before binning):")
        print(f"{'='*80}")
        print(f"Number of pre-frames: {len(new_extracted_frames)}")
        print(f"Values: {new_extracted_frames}")
        print(f"Non-zero count: {(new_extracted_frames > 0).sum()}")
        print(f"Mean: {new_extracted_frames.mean():.4f}")
        
        # Direct comparison
        print(f"\n{'='*80}")
        print("COMPARISON:")
        print(f"{'='*80}")
        
        if len(matlab_pre_values) == len(new_extracted_frames):
            print(f"Same number of frames: YES ({len(matlab_pre_values)})")
            print(f"Values match: {np.array_equal(matlab_pre_values, new_extracted_frames)}")
            if not np.array_equal(matlab_pre_values, new_extracted_frames):
                print(f"Max difference: {np.max(np.abs(matlab_pre_values - new_extracted_frames))}")
                print(f"Number of differing frames: {(matlab_pre_values != new_extracted_frames).sum()}")
        else:
            print(f"Different number of frames!")
            print(f"  MATLAB: {len(matlab_pre_values)}")
            print(f"  NEW: {len(new_extracted_frames)}")
            print(f"\nThis explains the difference - different frames are being extracted!")
            
else:
    print(f"\nFile not found at: {matlab_csv_path}")
    print("\nPlease provide the correct path to the MATLAB CSV file:")
    print("  post_outcome_main_datasets_neurons_trial activity timeseries data_<date>_timeseries.csv")

In [ ]:
matlab_row[[c for c in matlab_row.columns if "f_" not in c]]

In [ ]:
trial_4_pre[cell_col]

In [ ]:
matlab_csv.columns.tolist()

In [ ]:
config['timeseries']['pre_frames']

In [ ]:
# 1. Check if any cells were dropped in Python
test_subject = '13_4_WT_RS2'
subject_df = raster_dataframes[test_subject]

cell_cols = [c for c in subject_df.columns if c.startswith('cell_')]
print(f"Cells in Python raster: {len(cell_cols)}")
print(f"Cell numbers: {[int(c.replace('cell_','')) for c in cell_cols[:10]]}...")  # First 10

# 2. Check how many neurons MATLAB has for this subject
matlab_neurons = matlab_csv[matlab_csv['name'] == test_subject]['neuron_ID'].unique()
print(f"\nNeurons in MATLAB CSV: {len(matlab_neurons)}")
print(f"Neuron IDs: {sorted(matlab_neurons)[:10]}...")  # First 10

# 3. Check if cell_42 exists in Python
if 'cell_42' in cell_cols:
    print(f"\ncell_42 exists in Python: YES")
else:
    print(f"\ncell_42 exists in Python: NO - it may have been dropped!")
    
# 4. Check if neuron_ID 42 exists in MATLAB
if 42 in matlab_neurons:
    print(f"neuron_ID 42 exists in MATLAB: YES")
else:
    print(f"neuron_ID 42 exists in MATLAB: NO")

# 5. CRITICAL: Check if the neuron_id in the final NEW dataframe is correct
# After pivoting, the neuron_id is created as: subject_name + '-' + cell_number
new_neuron_ids = new_df[new_df['neuron_id'].str.startswith('13_4_WT_RS2')]['neuron_id'].unique()
print(f"\nNeuron IDs in NEW dataset for this subject: {len(new_neuron_ids)}")
print(f"Does '13_4_WT_RS2-42' exist? {'13_4_WT_RS2-42' in new_neuron_ids}")

# 6. Check if CANON uses 'unique_ID' that matches
canon_neuron_ids = canon_df[canon_df['unique_ID'].str.startswith('13_4_WT_RS2')]['unique_ID'].unique()
print(f"\nNeuron IDs in CANON for this subject: {len(canon_neuron_ids)}")
print(f"Does '13_4_WT_RS2-42' exist? {'13_4_WT_RS2-42' in canon_neuron_ids}")

In [ ]:
# For trial 4, check POST-OUTCOME activity at the raw level

# MATLAB CSV: Get post-outcome frame values
matlab_row = matlab_csv[(matlab_csv['name'] == test_subject) & 
                        (matlab_csv['neuron_ID'] == 42) &
                        (matlab_csv['trial_num'] == 4)]

post_frame_cols = sorted([col for col in matlab_csv.columns if col.startswith('f_') and not col.startswith('-f_')],
                         key=lambda x: int(x.split('_')[1]))

matlab_post_values = matlab_row[post_frame_cols[:60]].values[0]  # First 60 post frames

print("MATLAB CSV post-outcome (first 60 frames):")
print(matlab_post_values)
print(f"Non-zero: {(matlab_post_values > 0).sum()}")

# Python raw: Get post-outcome frame values for trial 4
trial_4_post = subject_df[(subject_df['trial_num'] == 4) & 
                           (subject_df['trial_section'] == 'post_outcome')]

python_post_values = trial_4_post['cell_42'].values[:60]  # First 60 post frames

print("\nPython raw post-outcome (first 60 frames):")
print(python_post_values)
print(f"Non-zero: {(python_post_values > 0).sum()}")

# Compare
print(f"\nValues match: {np.array_equal(matlab_post_values, python_post_values)}")
if not np.array_equal(matlab_post_values, python_post_values):
    print(f"Max difference: {np.max(np.abs(matlab_post_values - python_post_values))}")

In [ ]:
test_neuron = '13_4_WT_RS2-42'
test_stage = 'Early_IA_Error'
test_trial = 4

# Get NEW values with the updated extraction
new_row = new_df[(new_df['neuron_id'] == test_neuron) & 
                  (new_df['task_phase_vec'] == test_stage) &
                  (new_df['trial_num'] == test_trial)]

canon_row = canon_df[(canon_df['unique_ID'] == test_neuron) & 
                      (canon_df['task_phase_vec'] == test_stage) &
                      (canon_df['trial_num'] == test_trial)]

print("NEW pre-outcome bins (first 12):")
print(f"{new_ts_cols[:12]}")
print(new_row[new_ts_cols[:12]].values[0])

print("\nCANON pre-outcome bins (first 12):")
print(f"{canon_ts_cols[:12]}")
print(canon_row[canon_ts_cols[:12]].values[0])

print(f"\nValues match: {np.allclose(canon_row[canon_ts_cols].values, new_row[new_ts_cols].values, equal_nan=True)}")

In [ ]:
new_row

In [ ]:
canon_row

In [ ]:
# For trial 4, check what frames are being extracted by the new function
test_subject = '13_4_WT_RS2'
subject_df = raster_dataframes[test_subject]

trial_4_pre_all = subject_df[(subject_df['trial_num'] == 4) & 
                              (subject_df['trial_section'] == 'pre_outcome')]

print(f"Total pre-outcome frames: {len(trial_4_pre_all)}")

# After trimming to last 300
trial_4_pre_trimmed = trial_4_pre_all.tail(300)
print(f"After trim to 300: frames {trial_4_pre_all.index[0]} to {trial_4_pre_trimmed.index[0]} ... {trial_4_pre_trimmed.index[-1]}")

# After taking last 100
trial_4_pre_final = trial_4_pre_trimmed.tail(100)
print(f"After taking last 100: frames {trial_4_pre_final.index[0]} to {trial_4_pre_final.index[-1]}")

# Check activity in these final 100 frames
activity_values = trial_4_pre_final['cell_42'].values
print(f"\nActivity in extracted 100 frames:")
print(f"Non-zero count: {(activity_values > 0).sum()}")
print(f"Mean: {activity_values.mean()}")

# Show the actual position in the original 960 frames
first_idx = trial_4_pre_all.index.get_loc(trial_4_pre_final.index[0])
last_idx = trial_4_pre_all.index.get_loc(trial_4_pre_final.index[-1])
print(f"\nThese are frames {first_idx} to {last_idx} out of the 960 total pre-outcome frames")
print(f"Frame values at these positions:")
print(trial_4_pre_all['cell_42'].values[first_idx:last_idx+1])

In [ ]:
# Check what labels are in the 960 pre_outcome frames
trial_4_pre_all = subject_df[(subject_df['trial_num'] == 4) & 
                              (subject_df['trial_section'] == 'pre_outcome')]

print("Labels in the 960 'pre_outcome' frames:")
print(trial_4_pre_all['labels'].value_counts().sort_index())

# Check how many have labels 2-4 specifically
labels_2_to_4_count = ((trial_4_pre_all['labels'] >= 2) & (trial_4_pre_all['labels'] <= 4)).sum()
print(f"\nFrames with labels 2-4: {labels_2_to_4_count} out of 960")